# UzTerm AI — XLM-R bilan termin aniqlash (Google Colab)

Bu notebook transformer modelini (**XLM-RoBERTa base**) CRF bilan **aynan bir xil** sharoitda o'qitadi va solishtiradi:

- bir xil train / validation / test (`data/splits/manifest.json`);
- bir xil so'z tokenlashi va BIO teglar;
- bir xil metrikalar, bootstrap va yangi terminlar recall'i (`app.models.experiment`).

**Boshlashdan oldin:** `Runtime → Change runtime type → T4 GPU`. Keyin `Runtime → Run all`.

Vaqt: o'rnatish ~2 daqiqa, o'qitish ~10–15 daqiqa (T4).

> Natijalar **AI annotatsiyasi** (kumush maʼlumot) bo'yicha. Inson oltin standarti tayyor bo'lgach, notebook qayta ishga tushiriladi.

## 1. GPU tekshiruvi

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "GPU yo'q: Runtime -> Change runtime type -> T4 GPU"

## 2. Repozitoriyni yuklash

Repo **private** bo'lsa, GitHub token kerak: GitHub → Settings → Developer settings → Personal access tokens → *Fine-grained*, faqat shu repo uchun **Contents: Read-only**. So'ralganda tokenni kiriting (u notebookda saqlanmaydi). Repo public bo'lsa, bo'sh qoldirib Enter bosing.

In [ ]:
import getpass, os, subprocess

REPO = "Mirzokhidbek/graduatemdi"
BRANCH = "main"

token = getpass.getpass("GitHub token (public repo bo'lsa bo'sh qoldiring): ").strip()
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if not os.path.exists("/content/graduatemdi"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, "/content/graduatemdi"], check=True)
del token, url
%cd /content/graduatemdi
!git log --oneline -1

## 3. Kutubxonalar

Colab'da `torch` va `transformers` bor; CRF modellarini yuklash uchun `python-crfsuite` qo'shiladi.

In [ ]:
!pip install -q python-crfsuite==0.9.12 "transformers>=4.40" sentencepiece
import os, sys
os.environ["PYTHONPATH"] = "backend"
sys.path.insert(0, "backend")
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| GPU:", torch.cuda.is_available())

## 4. To'plamlarni tiklash

`data/processed/` gitda saqlanmaydi; u muzlatilgan manifestdan deterministik qayta yaratiladi.

In [ ]:
!python -m app.dataset.split data/annotations/claude/*.jsonl --include-drafts --output-dir data/processed/silver
!wc -l data/processed/silver/*.jsonl

## 5. O'qitish va test

- har epoxadan keyin validation exact F1 o'lchanadi, eng yaxshi epoxa saqlanadi (`models/transformer/`);
- test **bir marta** baholanadi;
- solishtirish: lug'at (train), CRF, CRF + gazetteer (repodagi saqlangan modellar).

Giperparametrlar: lr 3e-5, batch 8, 10 epoxa, warmup 10 %, weight decay 0.01, seed 13. Boshqa model uchun: `--model bert-base-multilingual-cased`.

In [ ]:
!python -m app.models.transformer --model xlm-roberta-base

## 6. Qayta belgilangan testda (claude2) — o'sha model, o'qitmasdan

In [ ]:
!python -m app.models.transformer --load --test-files data/annotations/claude2/*.jsonl --output reports/experiment-transformer-test2.json

## 7. Taqqoslash jadvali

In [ ]:
!python -m app.evaluation.report
from IPython.display import Markdown, display
text = open("docs/experiments-table.md", encoding="utf-8").read()
display(Markdown(text[text.index("## 6."):]))

## 8. Natijalarni yuklab olish

Arxivni repodagi o'sha joylarga qo'ying (`reports/`, `docs/`) va commit qiling. Model (~1 GB) gitga qo'shilmaydi.

In [ ]:
!zip -q transformer-results.zip reports/experiment-transformer.json reports/experiment-transformer-test2.json docs/experiments-table.md
from google.colab import files
files.download("transformer-results.zip")

## 9. Modelni ilovaga olish (ixtiyoriy, ~1 GB)

O'qitilgan model web ilovaning "Tahlil" sahifasida **XLM-R** usuli sifatida ishlaydi. Model Google Drive'ga
saqlanadi (to'g'ridan-to'g'ri yuklab olish 1 GB faylda ko'pincha uziladi). Drive'dan `uzterm-transformer.zip` ni
yuklab oling va repo ildizida oching: `unzip uzterm-transformer.zip` → `models/transformer/`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!cd /content/graduatemdi && zip -qr /content/drive/MyDrive/uzterm-transformer.zip models/transformer
!ls -lh /content/drive/MyDrive/uzterm-transformer.zip